In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

In [2]:
torch.manual_seed(42)

In [3]:
df = pd.read_csv('fmnist_small.csv')
df.head()

,label,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,pixel9,...,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783,pixel784
0,9,0,0,0,0,0,0,0,0,0,...,0,7,0,50,205,196,213,165,0,0
1,7,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,1,0,0,0,...,142,142,142,21,0,3,0,0,0,0
3,8,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,8,0,0,0,0,0,0,0,0,0,...,213,203,174,151,188,10,0,0,0,0


In [4]:
X = df.iloc[:,1:].values
y = df.iloc[:,0].values

In [5]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [6]:
X_train

array([[ 0,  0,  0, ...,  0,  0,  0],
       [ 0,  0,  0, ...,  0,  0,  0],
       [ 0,  0,  0, ...,  0,  0,  0],
       ...,
       [ 0,  0,  0, ...,  0,  0,  0],
       [ 0,  0,  0, ...,  0,  0,  0],
       [ 0,  0,  0, ..., 16,  0,  0]], shape=(4800, 784))

In [7]:
X_train = X_train/255.0
X_test = X_test/255.0

In [8]:
X_train

array([[0.       , 0.       , 0.       , ..., 0.       , 0.       ,
        0.       ],
       [0.       , 0.       , 0.       , ..., 0.       , 0.       ,
        0.       ],
       [0.       , 0.       , 0.       , ..., 0.       , 0.       ,
        0.       ],
       ...,
       [0.       , 0.       , 0.       , ..., 0.       , 0.       ,
        0.       ],
       [0.       , 0.       , 0.       , ..., 0.       , 0.       ,
        0.       ],
       [0.       , 0.       , 0.       , ..., 0.0627451, 0.       ,
        0.       ]], shape=(4800, 784))

In [9]:
# Create custom dataclass
class CustomDataset(Dataset):
    def __init__(self, features, labels):
        self.features = torch.tensor(features, dtype=torch.float32)
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.features)
    
    def __getitem__(self, index):
        return self.features[index], self.labels[index]

In [10]:
# Create train dataset object
train_dataset = CustomDataset(X_train, y_train)

In [11]:
test_dataset = CustomDataset(X_test, y_test)

In [12]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [13]:
# Define NN
class model(nn.Module):
    def __init__(self, feature_size):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(feature_size, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 10)
            # No need softmax. Softmax is by default implemented by cross entropy
        )

    def forward(self, x):
        return self.model(x)

In [14]:
# set learning rate and epochs
epochs = 100
lr = 0.1

In [15]:
# instatiate the model
model = model(X_train.shape[1])

# loss function
criterion = nn.CrossEntropyLoss()

# optimizer
optimizer = optim.SGD(model.parameters(), lr)

In [16]:
# Training loop
for epoch in range(epochs):
    total_epoch_loss = 0

    for batch_features, batch_label in train_loader:
        # forward
        outputs = model(batch_features)

        # loss
        loss = criterion(outputs, batch_label)

        optimizer.zero_grad()

        # backward
        loss.backward()

        # update grad
        optimizer.step()
    
        total_epoch_loss = total_epoch_loss + loss.item()

    print(f"Epoch: {epoch+1}, Loss: {total_epoch_loss/(len(train_loader))}")

Epoch: 1, Loss: 1.3216368504365286
Epoch: 2, Loss: 0.7793365436792373
Epoch: 3, Loss: 0.6427524662017823
Epoch: 4, Loss: 0.5751657338937124
Epoch: 5, Loss: 0.5278772577643395
Epoch: 6, Loss: 0.4953110004464785
Epoch: 7, Loss: 0.46192685594161353
Epoch: 8, Loss: 0.43552650665243464
Epoch: 9, Loss: 0.4189451836546262
Epoch: 10, Loss: 0.3993094977239768
Epoch: 11, Loss: 0.38615925828615827
Epoch: 12, Loss: 0.37421553740898766
Epoch: 13, Loss: 0.3471495986978213
Epoch: 14, Loss: 0.3478396603961786
Epoch: 15, Loss: 0.3135210007429123
Epoch: 16, Loss: 0.3120525466402372
Epoch: 17, Loss: 0.29320211564501125
Epoch: 18, Loss: 0.2869953137636185
Epoch: 19, Loss: 0.27722830668091775
Epoch: 20, Loss: 0.2597641822944085
Epoch: 21, Loss: 0.26642471527059874
Epoch: 22, Loss: 0.24103446503480275
Epoch: 23, Loss: 0.24018121351798374
Epoch: 24, Loss: 0.22060980891187987
Epoch: 25, Loss: 0.22015887394547462
Epoch: 26, Loss: 0.2125904703140259
Epoch: 27, Loss: 0.21707382981975873
Epoch: 28, Loss: 0.206910